## Agentic Loop
With the LLM in the driver's seat, we have an agent. It's an AI assistant whose goal is to help the user.

An agent has three parts:

    - Instructions, the role and behavior we want. We pass this as the developer message. The better the instructions, the better the agent helps.
    - Tools, the functions the agent can call to carry out the task. For us that's only search.
    - Memory, the message history. We append every prompt, every model output, and every tool result. The agent reads this to know what it has already tried.

Everything below is the code that wires these three together inside a loop.

In [1]:
# Importing libraries and create openai client
import os
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI
import requests

In [2]:
openai_client = OpenAI(
    api_key=os.getenv("LMSTUDIO_API_KEY"),
    base_url=os.getenv("LMSTUDIO_HOST")
)

model = "qwen/qwen3.5-9b"

In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [4]:
# Developer Prompt - we define how we want the LLM model to behave
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

🔍 SEARCH STRATEGY (IMPORTANT - FOLLOW THESE STEPS):

1. FIRST SEARCH: Use ALL keywords from the user question for the initial search.
   This will give you general information about the topic.

2. ANALYZE: Carefully read all results from the first search.
   
3. SECOND SEARCH: Based on what you found, perform a SECOND search with:
   - New keywords derived from the first results
   - More specific questions or follow-up queries
   - OR ask yourself: "What else should I search for to give a complete answer?"

4. SYNTHESIZE: Combine information from BOTH searches into a comprehensive answer.

5. ASK: End by asking if there are other areas the user wants to explore.

Make sure you perform at least 2 separate searches before answering.

Make sure you perform no more than 3 separate searches before answering.
""".strip()

question= "Can I still join the course?"

In [5]:
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

In [6]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [7]:
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"]
        }
    }
}

In [8]:
response = openai_client.chat.completions.create(
    model=model,
    messages=messages,
    tools=[search_tool],
    parallel_tool_calls=True,
)

In [9]:
response

ChatCompletion(id='chatcmpl-5gd3sswfzqa0ou7qixchvb', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='666799672', function=Function(arguments='{"query":"join course late enrollment"}', name='search'), type='function')], reasoning_content=''))], created=1782229106, model='qwen/qwen3.5-9b', object='chat.completion', moderation=None, service_tier=None, system_fingerprint='qwen/qwen3.5-9b', usage=CompletionUsage(completion_tokens=27, prompt_tokens=508, total_tokens=535, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None), prompt_tokens_details=None), stats={})

In [10]:
response.choices[0]

Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='666799672', function=Function(arguments='{"query":"join course late enrollment"}', name='search'), type='function')], reasoning_content=''))

In [11]:
response.choices[0].message.tool_calls[0]

ChatCompletionMessageFunctionToolCall(id='666799672', function=Function(arguments='{"query":"join course late enrollment"}', name='search'), type='function')

In [12]:
import json

call = response.choices[0].message.tool_calls[0]
args_dict = json.loads(call.function.arguments)

results = search(**args_dict)
result_json = json.dumps(results, indent=2)

In [13]:
call

ChatCompletionMessageFunctionToolCall(id='666799672', function=Function(arguments='{"query":"join course late enrollment"}', name='search'), type='function')

In [14]:
call.function.name

'search'

In [15]:
args_dict

{'query': 'join course late enrollment'}

In [16]:
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "bd31146b0e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "When will the course be offered next?",
    "answer": "Summer 2027."
  },
  {
    "id": "04919992b3",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "How should I start the course and follow the weekly workflow?",
    "answer": "Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).

In [17]:
# Initial message history
messages

[{'role': 'system',
  'content': 'You\'re a course teaching assistant.\nYou\'re given a question from a course student and your task is to answer it.\n\n🔍 SEARCH STRATEGY (IMPORTANT - FOLLOW THESE STEPS):\n\n1. FIRST SEARCH: Use ALL keywords from the user question for the initial search.\n   This will give you general information about the topic.\n\n2. ANALYZE: Carefully read all results from the first search.\n\n3. SECOND SEARCH: Based on what you found, perform a SECOND search with:\n   - New keywords derived from the first results\n   - More specific questions or follow-up queries\n   - OR ask yourself: "What else should I search for to give a complete answer?"\n\n4. SYNTHESIZE: Combine information from BOTH searches into a comprehensive answer.\n\n5. ASK: End by asking if there are other areas the user wants to explore.\n\nMake sure you perform at least 2 separate searches before answering.\n\nMake sure you perform no more than 3 separate searches before answering.'},
 {'role': 'us

I need output tool call info and the tool call output

In [18]:
# Put all message history together
# Need to append LLM's tool call info to messages history
messages.append({
    "role": "tool",
    "arguments": args_dict,
    "tool_call_id": call.id,
    "name": call.function.name,
    "content": '',
})


In [19]:
messages

[{'role': 'system',
  'content': 'You\'re a course teaching assistant.\nYou\'re given a question from a course student and your task is to answer it.\n\n🔍 SEARCH STRATEGY (IMPORTANT - FOLLOW THESE STEPS):\n\n1. FIRST SEARCH: Use ALL keywords from the user question for the initial search.\n   This will give you general information about the topic.\n\n2. ANALYZE: Carefully read all results from the first search.\n\n3. SECOND SEARCH: Based on what you found, perform a SECOND search with:\n   - New keywords derived from the first results\n   - More specific questions or follow-up queries\n   - OR ask yourself: "What else should I search for to give a complete answer?"\n\n4. SYNTHESIZE: Combine information from BOTH searches into a comprehensive answer.\n\n5. ASK: End by asking if there are other areas the user wants to explore.\n\nMake sure you perform at least 2 separate searches before answering.\n\nMake sure you perform no more than 3 separate searches before answering.'},
 {'role': 'us

In [20]:
# Need to send search results to messages history as well
messages.append({
    "role": "tool",
    "tool_call_id": call.id, # for the response API you use call.call_id
    "content": result_json,
})

In [21]:
# Updated message history
messages # this info will be sent to the LLM

[{'role': 'system',
  'content': 'You\'re a course teaching assistant.\nYou\'re given a question from a course student and your task is to answer it.\n\n🔍 SEARCH STRATEGY (IMPORTANT - FOLLOW THESE STEPS):\n\n1. FIRST SEARCH: Use ALL keywords from the user question for the initial search.\n   This will give you general information about the topic.\n\n2. ANALYZE: Carefully read all results from the first search.\n\n3. SECOND SEARCH: Based on what you found, perform a SECOND search with:\n   - New keywords derived from the first results\n   - More specific questions or follow-up queries\n   - OR ask yourself: "What else should I search for to give a complete answer?"\n\n4. SYNTHESIZE: Combine information from BOTH searches into a comprehensive answer.\n\n5. ASK: End by asking if there are other areas the user wants to explore.\n\nMake sure you perform at least 2 separate searches before answering.\n\nMake sure you perform no more than 3 separate searches before answering.'},
 {'role': 'us

In [22]:
# Run once more
response = openai_client.chat.completions.create(
    model=model,
    messages=messages,
    tools=[search_tool],
    parallel_tool_calls=True,
)

In [23]:
print(response.choices[0].finish_reason)

tool_calls


Now how do we make multiple tool calls with the LLM model?

Basic steps:
1. make a call to the LLM <--
2. LLM decided to invoke search('params')
3. We invoke the search, we have the results
4. Send the results back to the LLM (another call) <--
5. LLM processes the results
6. LLM makes an additional call to use search again* (create a tool call loop until LLM model is ready to give answer)
7. Send the results additional results to the LLM (another call) <--
8. LLM processes the results
9. LLM gives the answer

This is call Agentic Loop

Parts to construct agentic looping:
- We give a role to the agent which includes instructions to use specific tools and call multiples as needed
- We have memory which is represented by our messages history so far in order to keep track of the previous interactions
- Finally we have the tools so the agent knows what tools it can use to fulfill the request from the user

We also include an exit loop which can implemented by checking if there are no more tool calls.

In [52]:
# Developer Prompt - we define how we want the LLM model to behave
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

🔍 SEARCH STRATEGY (IMPORTANT - FOLLOW THESE STEPS):

1. FIRST SEARCH: Use ALL keywords from the user question for the initial search.
   This will give you general information about the topic.

2. ANALYZE: Carefully read all results from the first search.
   
3. SECOND SEARCH: Based on what you found, perform a SECOND search with:
   - New keywords derived from the first results
   - More specific questions or follow-up queries
   - OR ask yourself: "What else should I search for to give a complete answer?"

4. SYNTHESIZE: Combine information from BOTH searches into a comprehensive answer.

5. ASK: End by asking if there are other areas the user wants to explore.

Make sure you perform at least 2 separate searches before answering.
""".strip()

question= "Can I still join the course?"

In [53]:
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

In [26]:
response = openai_client.chat.completions.create(
    model=model,
    messages=messages,
    tools=[search_tool],
    parallel_tool_calls=True,
)

In [27]:
response

ChatCompletion(id='chatcmpl-ttigydqd65xyut0dsp6rl', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='882950005', function=Function(arguments='{"query":"join course enroll late registration deadline"}', name='search'), type='function')], reasoning_content=''))], created=1782229143, model='qwen/qwen3.5-9b', object='chat.completion', moderation=None, service_tier=None, system_fingerprint='qwen/qwen3.5-9b', usage=CompletionUsage(completion_tokens=29, prompt_tokens=493, total_tokens=522, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None), prompt_tokens_details=None), stats={})

In [102]:
# Creating a helper call search function repeatedly
def make_call(call):
    """Convert JSON string to dict and invoke the search function."""
    args = json.loads(call.function.arguments)
    
    if call.function.name == "search":
        results = search(**args)
    
    result_json = json.dumps(results, indent=2)

    return {
    "role": "tool",
    "tool_call_id": call.id,
    "content": result_json,
    }


This make call function will takes the args from the response call and inputs them into the search function, calls the search function and finally return the search results which will be sent to the LLM model.

note: chat.completions api may not be able to store multiple calls to one variable

In [29]:
for item in response.choices:
    print(item)

Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='882950005', function=Function(arguments='{"query":"join course enroll late registration deadline"}', name='search'), type='function')], reasoning_content=''))


In [30]:
for item in response.choices:
    print(item.message.tool_calls)

[ChatCompletionMessageFunctionToolCall(id='882950005', function=Function(arguments='{"query":"join course enroll late registration deadline"}', name='search'), type='function')]


In [31]:
for item in response.choices:
    print(item.message.tool_calls[0].function.name)

search


In [32]:
for item in response.choices:
    print(item.message.tool_calls[0].function.arguments)

{"query":"join course enroll late registration deadline"}


In [33]:
# Process all tool calls
for item in response.choices:
    print(f"Call response: {item}")
    if item.message.tool_calls[0].type == 'function':
        print(f"Processing tool call: {item.message.tool_calls[0].function.name}")

        call_output = make_call(item.message.tool_calls[0])

        messages.append({
            "role": "tool",
            "arguments": item.message.tool_calls[0].function.arguments,
            "tool_call_id": item.message.tool_calls[0].id,
            "name": item.message.tool_calls[0].function.name,
            "content": item.message.content
        })

        messages.append(call_output)

    if item.finish_reason == 'stop':
        pass


Call response: Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='882950005', function=Function(arguments='{"query":"join course enroll late registration deadline"}', name='search'), type='function')], reasoning_content=''))
Processing tool call: search


In [34]:
messages

[{'role': 'system',
  'content': 'You\'re a course teaching assistant.\nYou\'re given a question from a course student and your task is to answer it.\n\n🔍 SEARCH STRATEGY (IMPORTANT - FOLLOW THESE STEPS):\n\n1. FIRST SEARCH: Use ALL keywords from the user question for the initial search.\n   This will give you general information about the topic.\n\n2. ANALYZE: Carefully read all results from the first search.\n\n3. SECOND SEARCH: Based on what you found, perform a SECOND search with:\n   - New keywords derived from the first results\n   - More specific questions or follow-up queries\n   - OR ask yourself: "What else should I search for to give a complete answer?"\n\n4. SYNTHESIZE: Combine information from BOTH searches into a comprehensive answer.\n\n5. ASK: End by asking if there are other areas the user wants to explore.\n\nMake sure you perform at least 2 separate searches before answering.'},
 {'role': 'user', 'content': 'Can I still join the course?'},
 {'role': 'tool',
  'argume

In [35]:
response = openai_client.chat.completions.create(
    model=model,
    messages=messages,
    tools=[search_tool],
    parallel_tool_calls=True,
)

In [60]:
response.choices

[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Based on the FAQ search results, **yes, you can still join the course!**\n\nHere are the key details to keep in mind:\n\n*   **Registration:** You do not need a formal confirmation email or be on a specific registration list. According to one entry, "You\'re accepted" and you can just start learning.\n*   **Certification Caveat:** There is an important condition if you want a certificate. If you join late (or choose self-paced mode), you generally **cannot receive a certificate**. The FAQ states, "You don\'t get a certificate for the self-paced version of the course."\n*   **Why?** Certificates are only awarded if you complete the course as part of a "live cohort" where you can participate in peer reviews (specifically reviewing 3 capstone projects) after submitting your own project. This process happens at a specific time when the form is open for submissions, so joining later might exclude yo

In [ ]:
for item in response.choices:
    if item.message.tool_calls[0].type == 'function':
        print(f"Processing tool call: {item.message.tool_calls[0].function.name}")
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

### Creating the loop in one code

In [110]:
# Developer Prompt - we define how we want the LLM model to behave
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

🔍 SEARCH STRATEGY (IMPORTANT - FOLLOW THESE STEPS):

1. FIRST SEARCH: Use ALL keywords from the user question for the initial search.
   This will give you general information about the topic.

2. ANALYZE: Carefully read all results from the first search.
   
3. SECOND SEARCH: Based on what you found, perform a SECOND search with:
   - New keywords derived from the first results
   - More specific questions or follow-up queries
   - OR ask yourself: "What else should I search for to give a complete answer?"

4. SYNTHESIZE: Combine information from BOTH searches into a comprehensive answer.

5. ASK: End by asking if there are other areas the user wants to explore.

Make sure you perform at least 2 separate searches before answering.
""".strip()

question= "Can I still join the course?"

In [111]:
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

In [ ]:
it = 1

while True:
    print(f"Iteration #{it}...")
    has_function_calls = False

    response = openai_client.chat.completions.create(
        model=model,
        messages=messages,
        tools=[search_tool],
        parallel_tool_calls=True,
    )
    for item in response.choices:
        if item.finish_reason == 'tool_calls' and item.message.tool_calls[0].function.name == 'search':
            print(f"Processing tool call: {item.message.tool_calls[0].function.name}")
            print(f"query: {item.message.tool_calls[0].function.arguments}")
            call_output = make_call(item.message.tool_calls[0])
            messages.append(call_output)
            has_function_calls = True

        elif item.finish_reason == 'stop':
            print("ASSISTANT:")
            last_answer = item.message.content
            print(item.message.content)
    
    it = it + 1
    if has_function_calls == False:
        break

Iteration #1...
Processing tool call: search
Iteration #2...
Processing tool call: search
Iteration #3...
ASSISTANT:
Yes, you can still join the course! Based on the course FAQ:

**Joining the Course:**
- You can start whenever you want because the videos and GitHub materials are available.
- You don't even need to formally register; registration is just to gauge interest before the course starts date.
- You can simply start learning and submit homework while the submission form is open.

**Important Note About Certificates:**
If you are interested in receiving an official certificate, there are two important conditions:
1. You must join as part of a "live" cohort (not self-paced mode) because certificates require peer-reviewing other students' capstone projects, which can only happen when the course is actively running.
2. If you want a certificate, **you need to submit your project while we are still accepting submissions** for that cohort. If you miss this window, you won't be able 

In [108]:
item

Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="**Yes, you can still join the course!** 🎉\n\nAccording to the FAQ:\n- You're **accepted** just by registering (no confirmation email is even required).\n- You can start learning immediately and submit homework **while the registration form is open**.\n- Registration is mainly to gauge interest before the course starts.\n\n### Important note on certificates:\nIf you want a certificate, here’s what you need to keep in mind:\n- You must complete the capstone project while submissions are still being accepted (usually during a live cohort period).\n- Certificates aren’t available for self-paced mode — they require participating in a live cohort and completing peer reviews.\n\n### What should you do next?\n1. Register if needed (or just start learning).\n2. Follow the course materials via:\n   - [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/)\n   - General logistics docs\n   - 

In [105]:
response.choices[0].finish_reason

'stop'

In [122]:
# Wrapping entire loop in function
def agent_loop(instructions, question, model=model) -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"Iteration #{it}...")
        has_function_calls = False

        response = openai_client.chat.completions.create(
            model=model,
            messages=messages,
            tools=[search_tool],
            parallel_tool_calls=True,
        )
        
        for item in response.choices:
            if item.finish_reason == 'tool_calls' and item.message.tool_calls[0].function.name == 'search':
                print(f"Processing tool call: {item.message.tool_calls[0].function.name}")
                print(f"{item.message.tool_calls[0].function.arguments}")         
                call_output = make_call(item.message.tool_calls[0])
                messages.append(call_output)
                has_function_calls = True

            elif item.finish_reason == 'stop':
                print("ASSISTANT:")
                last_answer = item.message.content
                print(item.message.content)
        
        it = it + 1
        if has_function_calls == False:
            break
    
    return last_answer

In [117]:
agent_loop(instructions, "How do I run Olama locally?") # it works! :D

Iteration #1...
Processing tool call: search
query: {"query":"run Olama locally"}
Iteration #2...
Processing tool call: search
query: {"query":"run Olama locally docker"}
Iteration #3...
ASSISTANT:
Based on the search results, here is how you can run **Ollama locally**:

1.  **Install Ollama** first. The steps vary by operating system:
    *   **macOS**: Download the `.pkg` file and install it.
    *   **Windows**: Download the `.msi` installer.
    *   **Linux**: Run this command in your terminal:
        ```bash
        curl -fsSL https://ollama.com/install.sh | sh
        ```
    *   *(Source: FAQ ID `1d0b969028`)*

2.  **Run a Model** once installed, open a terminal and type:
    ```bash
    ollama run llama3
    ```
    This will download the model (~4GB) and start the local server. You can then chat with it directly in the terminal.

3.  **Verify the Server**: You can test if your local Ollama instance is running correctly by hitting this URL in your browser or via curl:
    ```b

'Based on the search results, here is how you can run **Ollama locally**:\n\n1.  **Install Ollama** first. The steps vary by operating system:\n    *   **macOS**: Download the `.pkg` file and install it.\n    *   **Windows**: Download the `.msi` installer.\n    *   **Linux**: Run this command in your terminal:\n        ```bash\n        curl -fsSL https://ollama.com/install.sh | sh\n        ```\n    *   *(Source: FAQ ID `1d0b969028`)*\n\n2.  **Run a Model** once installed, open a terminal and type:\n    ```bash\n    ollama run llama3\n    ```\n    This will download the model (~4GB) and start the local server. You can then chat with it directly in the terminal.\n\n3.  **Verify the Server**: You can test if your local Ollama instance is running correctly by hitting this URL in your browser or via curl:\n    ```bash\n    curl http://localhost:11434\n    ```\n    *(Source: FAQ ID `1d0b969028`)*\n\n**Running the Course Locally:**\nYou can run the entire course locally instead of using Codes

In [ ]:
# Restricting off topic questions - updating instructions
agent_loop(instructions, "what's queen gambit?")

Iteration #1...
Processing tool call: search
query: {"query":"Queen's Gambit what is it"}
Iteration #2...
Processing tool call: search
query: {"query":"queen gambit chess game history meaning"}
Iteration #3...
ASSISTANT:
I searched the FAQ database using the terms "queen gambit", but did not find any specific entries related to this topic. The search results returned information about other technical topics like LLM Zoomcamp projects, vector search, embeddings, and RAG (Retrieval-Augmented Generation), which are unrelated to the "Queen's Gambit."

Based on my general knowledge, I can provide you with the following information:

**The Queen's Gambit** is a popular term used in chess, but there are two main things it might refer to:

1. **The Chess Opening**: In chess, the Queen's Gambit (or Queen's Pawn Opening) is a well-known opening variation where White offers a pawn on d4 as a "gambit" - a strategic sacrifice to gain position or control in the game. The name comes from the historic

'I searched the FAQ database using the terms "queen gambit", but did not find any specific entries related to this topic. The search results returned information about other technical topics like LLM Zoomcamp projects, vector search, embeddings, and RAG (Retrieval-Augmented Generation), which are unrelated to the "Queen\'s Gambit."\n\nBased on my general knowledge, I can provide you with the following information:\n\n**The Queen\'s Gambit** is a popular term used in chess, but there are two main things it might refer to:\n\n1. **The Chess Opening**: In chess, the Queen\'s Gambit (or Queen\'s Pawn Opening) is a well-known opening variation where White offers a pawn on d4 as a "gambit" - a strategic sacrifice to gain position or control in the game. The name comes from the historical belief that it was an opening designed for queens, though this isn\'t technically accurate.\n\n2. **The Netflix Series**: More likely, you\'re referring to the hit Netflix series "Queen\'s Gambit" (2020), st

In [125]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

🔍 SEARCH STRATEGY (IMPORTANT - FOLLOW THESE STEPS):

1. FIRST SEARCH: Use ALL keywords from the user question for the initial search.
   This will give you general information about the topic.

2. ANALYZE: Carefully read all results from the first search.
   
3. SECOND SEARCH: Based on what you found, perform a SECOND search with:
   - New keywords derived from the first results
   - More specific questions or follow-up queries
   - OR ask yourself: "What else should I search for to give a complete answer?"

4. SYNTHESIZE: Combine information from BOTH searches into a comprehensive answer.

5. ASK: End by asking if there are other areas the user wants to explore.

Make sure you perform at least 2 separate searches before answering.

The question has to be about the course or its logistics, off topic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

Iteration #1...
Processing tool call: search
{"query":"queen gambit"}
Iteration #2...
ASSISTANT:
The search returned no results, which suggests that "Queen Gambit" is not covered in the course FAQ database as it seems to be a topic from chess or general culture rather than specific to the course curriculum. Based on the instruction that if the search returns nothing for questions about logistics or the course itself, it's likely an off-topic question, I cannot provide a detailed answer using the FAQ database.

However, just to clarify: The Queen's Gambit is a famous opening in chess where White plays 1. d4 and then offers a pawn at e2-e4 (or sometimes c2-c3 depending on notation). If you're asking about something specific within your course context that relates to this term, feel free to provide more details!

Are there other areas related to the course or logistics you'd like to explore?


'The search returned no results, which suggests that "Queen Gambit" is not covered in the course FAQ database as it seems to be a topic from chess or general culture rather than specific to the course curriculum. Based on the instruction that if the search returns nothing for questions about logistics or the course itself, it\'s likely an off-topic question, I cannot provide a detailed answer using the FAQ database.\n\nHowever, just to clarify: The Queen\'s Gambit is a famous opening in chess where White plays 1. d4 and then offers a pawn at e2-e4 (or sometimes c2-c3 depending on notation). If you\'re asking about something specific within your course context that relates to this term, feel free to provide more details!\n\nAre there other areas related to the course or logistics you\'d like to explore?'

This represent a light guardrail you can put in place to limit the scope of what responses can be provided. A real guardrail checks the input before the agent runs and can block off-topic questions outright.